# SRQ-FLY Priority 1: update, CUDA peak memory, and six-way ablation
This is a train-only feasibility gate. It never loads `test.pt`. Run cells in order on a T4 GPU and return the final ZIP for audit.

In [ ]:
# === Edit repository/path values only. Do not edit protocol or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/srq_priority1_cifar_features'
LARGE_WTA_CACHE = '/content/srq_priority1_wta_10000'
MATCHED_WTA_CACHE = '/content/srq_priority1_wta_4409'
SYSTEM_OUTPUT = '/content/srq_priority1_system'
ABLATION_OUTPUT = '/content/srq_priority1_ablation'
BATCH_SIZE = 128
NUM_WORKERS = 2

In [ ]:
# Fresh clone, install, GPU check, and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
SYSTEM_CONFIG='configs/srq_fly_update_optimization_fly10000.json'
SYSTEM_RUNNER='tools/srq_fly_system_benchmark.py'
ABLATION_CONFIG='configs/srq_fly_priority1_cifar100_train_only.json'
ABLATION_RUNNER='tools/srq_fly_priority1_ablation.py'
assert sha(SYSTEM_CONFIG)=='617ca4268fba9cd21610b1fe2e3d656afbfdb3375dbbbf356803a6cb6d336aa9'
assert sha(SYSTEM_RUNNER)=='fb8c98489a20c996e10cb43cd1454ece000a4c9f6659c5c8acf1942f212df3eb'
assert sha(ABLATION_CONFIG)=='7fff2ef42c7a9b41b8a12f48fb8c45604237fe7fed070a08bb4457a9949cd4ee'
assert sha(ABLATION_RUNNER)=='b65ce01bfc2e2f9f07a61ecd056637156b504626c74c45b6aa3a7c8162e22136'
assert sha('methods/srq_fly_optimized/learner.py')=='1cc61f63dccdd1ae9ec96f0821363588b1e76681bc8a45bbf02db8313476f305'
assert sha('methods/srq_fly_optimized/storage.py')=='99c4dab6f5c9e3da249f8c4dbf5cb9e5e303d8fd59f11e9d184db0d180bdf330'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
print('GPU:',torch.cuda.get_device_name(0))
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
# Synthetic correctness and state-contract gate.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_fly_optimized.py','tests/test_srq_fly_priority1_ablation.py','tests/test_srq_fly_learner.py','tests/test_srq_fly_math.py']
completed=subprocess.run(command)
assert completed.returncode==0,'Correctness gate failed; return the complete traceback.'
print('SRQ-FLY PRIORITY-1 CORRECTNESS GATE: PASS')

In [ ]:
# Six fresh processes: Exact FLY, locked SRQ, and four optimized backends.
# Each TASK line reports pure analytic-update time and PyTorch CUDA peak bytes.
system_result=Path(SYSTEM_OUTPUT)/'system_benchmark.json'
command=[sys.executable,'-u',SYSTEM_RUNNER,'run','--config',SYSTEM_CONFIG,'--output-dir',SYSTEM_OUTPUT,'--device','cuda']
print('SYSTEM BENCHMARK START: width=10000, 6 isolated workers',flush=True)
completed=subprocess.run(command)
assert system_result.is_file(),'System runner failed before writing its diagnostic JSON.'
system=json.loads(system_result.read_text())
import pandas as pd
display(pd.DataFrame(system['results'])[['method','total_update_seconds','persistent_state_bytes','peak_cuda_allocated_bytes','peak_cuda_reserved_bytes','solver_relative_residual']])
print('GATES:',json.dumps(system['gates'],indent=2))
print('blocked/exact update ratio:',system['optimized_blocked_qr_update_ratio_to_exact_fly'])
print('speedups:',json.dumps(system['speedup_over_locked'],indent=2))
blocked=next(row for row in system['results'] if row['method']=='optimized_blocked_qr_srq_int8')
print('blocked stage profile:',json.dumps(blocked['profiled_task_stage_seconds'],indent=2))
if completed.returncode!=0 or system['status']!='pass':
    raise RuntimeError('System benchmark gate failed. Stop; do not relax gates. The complete diagnostics are printed above.')
print('SYSTEM BENCHMARK PASS')

In [ ]:
# Download the exact checkpoint and processed CIFAR-100 artifact.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('checkpoint:',CHECKPOINT_PATH)
print('CIFAR-100:',CIFAR_ROOT)

In [ ]:
# Reuse or extract TRAIN features only; held-out test features stay absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_priority1','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START; progress is printed by batch/task.',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Six fresh processes on one locked CIFAR train/validation stream.
command=[sys.executable,'-u',ABLATION_RUNNER,'run','--config',ABLATION_CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--large-code-cache-dir',LARGE_WTA_CACHE,'--matched-code-cache-dir',MATCHED_WTA_CACHE,'--system-benchmark-result',str(system_result),'--output-dir',ABLATION_OUTPUT,'--device','cuda']
print('TRAIN-ONLY ABLATION START: 6 isolated methods; wait for TASK/DONE lines.',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'Ablation process failed; return the complete traceback.'
ablation=json.loads((Path(ABLATION_OUTPUT)/'priority1_results.json').read_text())
print('PRIORITY-1 DECISION:',ablation['status'])
print(json.dumps(ablation['gates'],indent=2))

In [ ]:
# Compact tables and plots; validation accuracy is not a paper test result.
import pandas as pd, matplotlib.pyplot as plt
rows=pd.DataFrame(ablation['results'])
display(rows[['method','validation_average_accuracy','persistent_state_bytes','peak_cuda_allocated_bytes','peak_cuda_reserved_bytes','maximum_solver_relative_residual']])
fig,axes=plt.subplots(1,3,figsize=(17,4.5))
axes[0].bar(rows.method,rows.validation_average_accuracy); axes[0].set_ylabel('Train-validation AIA (%)'); axes[0].tick_params(axis='x',rotation=70)
axes[1].bar(rows.method,rows.persistent_state_bytes/2**20); axes[1].set_ylabel('Persistent state (MiB)'); axes[1].tick_params(axis='x',rotation=70)
axes[2].bar(rows.method,rows.peak_cuda_allocated_bytes/2**30); axes[2].set_ylabel('Peak CUDA allocated (GiB)'); axes[2].tick_params(axis='x',rotation=70)
fig.tight_layout(); fig.savefig('/content/srq_fly_priority1_summary.png',dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# Export evidence only; feature/WTA caches are deliberately excluded.
bundle=Path('/content/srq_fly_priority1_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(SYSTEM_OUTPUT,bundle/'system')
shutil.copytree(ABLATION_OUTPUT,bundle/'ablation')
shutil.copy2('/content/srq_fly_priority1_summary.png',bundle/'summary.png')
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)